<a href="https://colab.research.google.com/github/fariha34/CSE-427/blob/main/RandomForestlab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#importing the necessary libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_link = 'https://drive.google.com/file/d/1wADS0ybifgQJcPEtzqDurKAi_CEmwKDF/view?usp=sharing' # the file access must have to be Public


id = file_link.split("/")[-2]
# creating a new link using the id so that we can easily read the csv file in pandas
new_link = f'https://drive.google.com/uc?id={id}'
print(new_link)
df = pd.read_csv(new_link)

# let's look at the first few instances
df.head(10)

https://drive.google.com/uc?id=1wADS0ybifgQJcPEtzqDurKAi_CEmwKDF


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0
5,6,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,0
6,7,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,0
7,8,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S,0
8,9,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S,1
9,10,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,1


In [ ]:
class Node:
    """Helper class to represent a node in the tree."""
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature      # index
        self.threshold = threshold
        self.left = left            # left subtree
        self.right = right          # right subtree (
        self.value = value

    def is_leaf(self):
        return self.value is not None

###TASK 1

In [ ]:
class DecisionTree:
    def __init__(self, max_depth = None):
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y, depth = 0):
        if len(np.unique(y)) == 1 or (self.max_depth is not None and depth >= self.max_depth) or len(y) < 2:
            leaf_value = self._majority_class(y)
            node = Node(value=leaf_value)

            if depth == 0:   ## return to root
                self.root = node
            return node


        best_feature,best_threshold = self.find_best_split(X, y)
        if best_feature is None:
            leaf_value = self._majority_class(y)

            node = Node(value=leaf_value)
            if depth == 0:
                self.root = node
            return node

        left_mask = X[:, best_feature] <= best_threshold
        right_mask = ~left_mask


        if left_mask.sum() == 0 or right_mask.sum() == 0:
            leaf_value = self._majority_class(y)
            node = Node(value=leaf_value)
            if depth == 0:
                self.root = node
            return node


        left_node = self.fit(X[left_mask], y[left_mask], depth + 1)
        right_node = self.fit(X[right_mask], y[right_mask], depth + 1)

        node = Node(feature=best_feature, threshold=best_threshold,
                    left=left_node, right=right_node)
        if depth == 0:
            self.root = node
        return node


    def find_best_split(self, X, y):
        n_samples,n_features = X.shape
        best_gini = float('inf')
        best_feature=None
        best_threshold =None

        for feature in range(n_features):
            thresholds =np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask =X[:,feature] <= threshold
                right_mask = ~left_mask

                if left_mask.sum()== 0 or right_mask.sum()== 0:
                    continue

                gini = self._weighted_gini(y[left_mask], y[right_mask])

                if gini < best_gini:
                    best_gini = gini
                    best_feature = feature
                    best_threshold = threshold

        return best_feature,best_threshold


    def gini_impurity(self,y):
        if len(y) == 0:
            return 0
        _,counts = np.unique(y, return_counts=True)
        probabilities = counts / len(y)
        gini_impu=(1 - np.sum(probabilities ** 2))
        return gini_impu

    def _weighted_gini(self, y_left, y_right):
        n = len(y_left) + len(y_right)    #total sample n
        gini_left = self.gini_impurity(y_left)
        gini_right = self.gini_impurity(y_right)
        weighted_average = (
            (len(y_left) /n) * gini_left +
            (len(y_right) /n) * gini_right)
        return weighted_average

    def _majority_class(self, y):
        values, counts = np.unique(y, return_counts=True)
        return values[np.argmax(counts)]

    def predict(self, node, x):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self.predict(node.left, x)
        else:
            return self.predict(node.right, x)


# converting Pandas dataframe to NumPy Array
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy()
X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy()


# Create the model and fit the training data
decision_tree = DecisionTree(max_depth=5)
final_tree = decision_tree.fit(X_train_np, y_train_np) # returns the information of the final tree in a suitable data structure

# Predictions on the test set
y_pred_dt_scr = np.array([decision_tree.predict(final_tree, instance) for instance in X_test_np])

accuracy_dt_scr = 0
decision_tree = DecisionTree(max_depth=5)
final_tree = decision_tree.fit(X_train_np, y_train_np)

y_pred_dt_scr = np.array([decision_tree.predict(final_tree, instance) for instance in X_test_np])

accuracy_dt_scr = np.sum(y_pred_dt_scr == y_test_np) / len(y_test_np)

print("Decision Tree Classifier Accuracy:", accuracy_dt_scr)

Decision Tree Classifier Accuracy: 0.782051282051282


###TASK 2

In [ ]:
class RandomForest:
    def __init__(self, n_estimators = 100, max_depth = None):
      self.tress=[]
      self.n_estimators=n_estimators
      self.max_depth=max_depth

    def fit(self, X, y):
      self.trees=[]
      n_samples=X.shape[0]

      # Loop upto n_estimators:
      for _ in range(self.n_estimators):
          #Create a bootstrap sample (X_sample, y_sample) from the training data
          indices = np.random.choice(n_samples, size=n_samples, replace=True)
          X_sample, y_sample = X[indices], y[indices]

          dt_model = DecisionTree(max_depth=self.max_depth)
          tree_root = dt_model.fit(X_sample, y_sample)
          self.trees.append((dt_model, tree_root))


    def predict(self, X):

      tree_predictions = []
      for dt_model, tree_root in self.trees:
          predictions_for_this_tree = [dt_model.predict(tree_root, instance) for instance in X]
          tree_predictions.append(predictions_for_this_tree)

      tree_predictions = np.array(tree_predictions)


      final_predictions = []
      for sample_index in range(X.shape[0]):
          votes_sample = tree_predictions[:, sample_index]
          majority_class = self._majority_vote(votes_sample)
          final_predictions.append(majority_class)

      return np.array(final_predictions)

    def _majority_vote(self, votes):
        values, counts = np.unique(votes, return_counts=True)
        return values[np.argmax(counts)]

# Create the model and fit the training data
random_forest = RandomForest(n_estimators=25, max_depth=5)
random_forest.fit(X_train_np, y_train_np)

# Predictions on the test set
y_pred_rf_scr = random_forest.predict(X_test_np)


accuracy_rf_scr = np.sum(y_pred_rf_scr == y_test_np) / len(y_test_np)


print("Random Forest Classifier Accuracy:", accuracy_rf_scr)

Random Forest Classifier Accuracy: 0.782051282051282
